# CVaR

## 1. Data Preparation

Conditional Value at Risk (CVaR) estimates **tail severity**: once returns enter the worst region of the distribution, CVaR measures the average loss inside that tail. In Libélula, this gives a direct proxy for downside stress used by the probabilistic risk envelope.

### Importation and application of `setup()`

Load the standardized dataset splits with `setup()` so CVaR features are aligned with the same train/validation/test protocol used by the other envelope tools.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))

from setup import setup

data = setup(
    csv_path="../../data/processed/financial_tools_datset.csv",
    target_col="Price",
    horizon=1
)

import numpy as np
import pandas as pd

### Extraction of return arrays

We isolate `y_train`, `y_val`, and `y_test` because CVaR is computed over return distributions (not prices).

In [2]:
y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

## 2. Model Construction

CVaR does not train a predictive model here; instead, we construct a **distributional risk operator** that maps returns to tail-risk estimates.

**Conditional Value at Risk (CVaR) is a measure of extreme risk.**\
Imagine we look at many daily returns of an asset. Some days are normal, but some days are very bad. CVaR tries to answer the question:
>"If things go really badly, how bad do they get on average?"

The function works in three steps:
1. Find the VaR (Value at Risk)\
The VaR is a threshold that separates the worst outcomes from the rest of the distribution.
For example, the 5% VaR means that 5% of the worst returns fall below this threshold.
2. Select only tail observations below VaR.
3. Compute their mean to obtain CVaR.

For the risk envelope, this becomes a robust downside feature that is sensitive to crash-like regimes.

In [3]:
def compute_cvar(returns, alpha=0.05):
    """
    returns: array de retornos
    alpha: nível de risco (ex: 0.05 = 95% CVaR)
    """
    
    var = np.quantile(returns, alpha)

    tail_losses = returns[returns <= var]

    if len(tail_losses) == 0:
        return var

    cvar = tail_losses.mean()

    return cvar

### Tail-risk calculation on training split

In this step we calculate CVaR using the training data.

Two risk levels are computed:

- CVaR 95%\
This measures the average loss inside the worst 5% of returns.
- CVaR 99%\
This measures the average loss inside the worst 1% of returns, which represents more extreme situations.

This gives us a basic idea of how dangerous the asset can be in very bad market conditions and defines baseline tail-severity anchors for envelope interpretation.

In [4]:
cvar_95 = compute_cvar(y_train, alpha=0.05)
cvar_99 = compute_cvar(y_train, alpha=0.01)

print("CVaR 95:", cvar_95)
print("CVaR 99:", cvar_99)

CVaR 95: -0.01060433623406335
CVaR 99: -0.015076727943026325


## 3. Sanity Checks

Validate whether CVaR behaves consistently with risk theory and whether magnitudes are stable across splits.

### Check 1: Var vs. CVaR
This step performs a sanity check.\
We compare two values:\
- VaR (Value at Risk)
- CVaR (Conditional Value at Risk)

The important rule is:\
CVaR should always be worse (more negative) than VaR.

Why?

Because:

- VaR is just a threshold.
- CVaR is the average of the losses beyond that threshold.

So CVaR must represent deeper losses.

If CVaR is not worse than VaR, then something is wrong in the implementation.

> CVaR should always be worse than VaR

In [5]:
var_95 = np.quantile(y_train, 0.05)

print("VaR 95:", var_95)
print("CVaR 95:", cvar_95)

VaR 95: -0.007882198594690248
CVaR 95: -0.01060433623406335


**Perfect: -0.01 < -0.007**

### Check 2: Stability between datasets

This step checks whether the risk level is stable across different datasets.
>**We want to make sure the market does not look completely different in each dataset.**

The CVaR is calculated for:
1. Training data
2. Validation data
3. Test data

If these values are very different, it may indicate that:
- the market regime changed
- the dataset distribution is unstable
- the model may not generalize well

If the numbers are relatively similar, it means the dataset behaves consistently.

In [6]:
print("Train CVaR:", compute_cvar(y_train))
print("Val CVaR:", compute_cvar(y_val))
print("Test CVaR:", compute_cvar(y_test))

Train CVaR: -0.01060433623406335
Val CVaR: -0.008073579285988427
Test CVaR: -0.009332211234611193


## 4. Generation of Risk Features

Build a rolling CVaR time series so the envelope receives **time-varying downside risk features** instead of a single static scalar.

The function below calculates a rolling CVaR.
Instead of computing risk once for the whole dataset, we calculate risk over time.\
The idea is simple:
1. Take a window of past returns (for example, the last 100 observations).
2. Compute CVaR using only those observations.
3. Move the window forward one step.

Repeat the process for the entire dataset.

This produces a time series of risk estimates. **At every moment in time, we estimate how dangerous the market currently is.**

In [7]:
def rolling_cvar(returns, window=100, alpha=0.05):

    cvar_series = []

    for i in range(len(returns)):
        if i < window:
            cvar_series.append(np.nan)
            continue

        window_returns = returns[i-window:i]

        cvar = compute_cvar(window_returns, alpha)

        cvar_series.append(cvar)

    return np.array(cvar_series)

### Rolling feature assembly

Here we generate the rolling CVaR series.

First, we combine all returns:
- training
- validation
- test

This creates a single continuous sequence of returns. Then we apply the rolling CVaR function.

The output is an array containing a risk estimate for each point in time.

In the Libélula probabilistic envelope, this feature represents downside-tail pressure for RL decision making.

In [8]:
cvar_rolling = rolling_cvar(
    np.concatenate([y_train, y_val, y_test]),
    window=100,
    alpha=0.05
)

print("CVaR rolling shape:", cvar_rolling.shape)

CVaR rolling shape: (1367,)


## 5. Output Standardization

The resulting rolling CVaR vector is the standardized CVaR risk feature to be merged with other envelope signals in the RL dataset.